# Attention Mechanism

The following Source its from Karpathy based code.

https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=2Num7sX9CKOH

In [3]:
from jinja2.optimizer import optimize
from zmq.decorators import context

# from Extras import logits
# download first dataset to make first experiments
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [4]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [5]:
print("lenght of dataset characters ", len(text), "and", type(text))

lenght of dataset characters  1115394 and <class 'str'>


In [6]:
# The first 1000 characters are this:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [25]:
# here are all the unique characters that occur in this text
unique_chars = set(text)
chars = sorted(list(unique_chars))
vocab_size = len(chars)
print("chars: ", chars)
print("vocal_size:", vocab_size)
print("unique_chars:", type(unique_chars))

chars:  ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
vocal_size: 65
unique_chars: <class 'set'>


In [8]:
# creating a mapping from characters to integer
string_to_integer = { ch:i   for i, ch in enumerate(chars) } # create a dictionary with character(ch):value(i)
integer_to_string = { i:ch   for i, ch in enumerate(chars) } # create a dictionary with value(ch):character(i)

In [9]:
# explanation of lambda`s:
# lambda s -> creates an anonymous function that takes one parameter s
#   and iterates through each c character in the string s, applying string_to_integer
encode = lambda s: [string_to_integer[c] for c in s]
decode = lambda l: ''.join(integer_to_string[i] for i in l)

In [10]:
# testing the encoding-decoding
print(encode("hii there"))
print(decode(encode("hii there")))# testing encode-decode way

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [11]:
import torch

In [12]:
data = torch.tensor(encode(text), dtype=torch.long) # tensor 64-bit signed integer

print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [13]:
# split data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [14]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [15]:
# lets see the structure of context -> prediction logic
x = train_data[:block_size+1]
y = train_data[1:block_size+1]
for t in range(block_size): #0, 1, 2, 3 ,4 , ...
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context}, target is {target}")

when input is tensor([18]), target is 47
when input is tensor([18, 47]), target is 56
when input is tensor([18, 47, 56]), target is 57
when input is tensor([18, 47, 56, 57]), target is 58
when input is tensor([18, 47, 56, 57, 58]), target is 1
when input is tensor([18, 47, 56, 57, 58,  1]), target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58


In [21]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel
block_size = 8 # what is the maximum context length for predictions

def get_batch(split):
    # e.g. data = torch.tensor([0,1,2,3,4,5,6,7,8,9])
    # ix = torch.tensor([0,2,4])
    # block_size = 3
    # batch_size = 3
    # ---->  get_batch() out: ------>
    # x = tensor([  [0,1,2],
    #               [2,3,4],
    #               [4,5,6] ])
    #    y = [[1,2,3],
    #         [3,4,5],
    #         [5,6,7]]

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) #(limit of generated vectors, block size (spaces) per vector)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [22]:
xb, yb = get_batch("train")
print("inputs:")
print(xb)
print("targets:")
print(yb.shape)
print(yb)
print("---------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()}, target is {target}")

inputs:
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
---------
when input is [24], target is 43
when input is [24, 43], target is 58
when input is [24, 43, 58], target is 5
when input is [24, 43, 58, 5], target is 57
when input is [24, 43, 58, 5, 57], target is 1
when input is [24, 43, 58, 5, 57, 1], target is 46
when input is [24, 43, 58, 5, 57, 1, 46], target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43], target is 39
when input is [44], target is 53
when input is [44, 53], target is 56
when input is [44, 53, 56], target is 1
when input is [44, 53, 56, 1], target is 58
when input is [44, 53, 56, 1, 58], target is 46
when input is [44, 53, 56, 1, 58, 46], target is

Take care of:
"logits" it's a specific term in machine learning.
What "logits" means:
Logits = Log-odds = Raw, unbounded output values from a neural network before applying softmax.
 - Etymology: "Logit" comes from "log" (logarithm) + "it" (from "unit"). Originally from statistics: logit function = log(p/(1-p)) where p is probability

It`s a value without normalization, pre activation of softmax or sigmoid.

In [23]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # vocab_size = 5
        # (cantidad de tokens unicos vocab, size vector salida) ------> nn.Embedding(voc_siz, voc_siz) -------> (crea una matriz aleatoria de pesos)
        # tokens = torch.tensor([0,2,4])
        # embeddings = embedding(tokens)
        # token 0 -> [logit00, logit01, logit02, logit03, logit04]
        # token 2 -> [logit20, logit21, logit22, logit23, logit24]
        # token 4 -> [logit40, logit41, logit42, logit43, logit44]
        # Internamente crea:
        # self.token_embedding_table.weight = torch.randn(65, 65) y estos son los parametros que se entrenan
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        """"

        """
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            # 3D tensor, e.g.
            # logits = torch.randn(32, 128, 50257)
            # B, T, C = logits.shape
            # ----> Batch size: B = 32
            # ----> Sequence lenght: T = 128
            # ----> Vocab size: C = 50257
            B, T, C = logits.shape

            # Reshape from 3D to 2D for cross_entropy
            # Before: (B, T, C) = (32, 128, 50257)
            #   - 32 sequences
            #   - Each sequence has 128 tokens
            #   - Each token has 50257 logits (one per vocab word)
            #
            # After: (B*T, C) = (4096, 50257)
            #   - Flattens batch and time dimensions
            #   - Treats each position as independent prediction
            #   - Now we have 4096 predictions, each with 50257 options
            #
            # Example with small numbers:
            # If B=2, T=3, C=5:
            #   Before reshape: [[[logits_for_token_0_batch_0],
            #                      [logits_for_token_1_batch_0],
            #                      [logits_for_token_2_batch_0]],
            #                     [[logits_for_token_0_batch_1],
            #                      [logits_for_token_1_batch_1],
            #                      [logits_for_token_2_batch_1]]]
            #   After reshape: [[logits_batch0_token0],
            #                   [logits_batch0_token1],
            #                   [logits_batch0_token2],
            #                   [logits_batch1_token0],
            #                   [logits_batch1_token1],
            #                   [logits_batch1_token2]]
            #   Shape: (6, 5)
            logits = logits.view(B*T, C)

            # (batch*seq_len, )
            targets = targets.view(B*T)

            # see extras_attention.ipynb 1.
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        Toma una secuencia inicial y predice los siguientes tokens uno a la vez.
        """
        for _ in range(max_new_tokens):
            # get predictions
            logits, loss = self.forward(idx)

            logits = logits[:, -1, :] # becomes (B, C)
            probs = F.softmax(logits, dim=-1) # get probs B, C

            # aleatory selection of tokens based in probability
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # appends the newly generated token to the end of existing sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

In [26]:
m  = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


starting_context = torch.zeros((1, 1), dtype=torch.long)
# [[0]]

new_tokens = m.generate(idx=starting_context, max_new_tokens=100)
# Shape: (1, 101), Content: [[0, 234, 67, 891, ...]]

extracted_sequence = new_tokens[0].tolist()
# Shape: (101,), Content: [0, 234, 67, 891, ...] ----> [0, 234, 67, 891, 12, 445, ...]

decode_to_text = decode(extracted_sequence)
# "Hello world! This is generated text from the model..."

print(decode_to_text)

torch.Size([32, 65])
tensor(4.6911, grad_fn=<NllLossBackward0>)

hfGb'zUQdcZLKdvtEz&kbEaSwsZVGvth'UJSCRsbx'UCCD?SHjxma.Xx$oHV!h&ER;,OSLUOnP!VVvXzz&spDICnZIf?zndBnvKw


In [30]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
# m.parameters(): Devuelve un generador (iterador)

In [31]:
batch_size = 32
for steps in range(100):

    # xb -> (32, block_size)
    # yb -> (32, block_size)
    xb, yb = get_batch("train")

    # model predict, compare token for each position
    logits, loss = m(xb, yb)

    # clear gradient from previous step
    # python accumulates gradients by default
    optimizer.zero_grad(set_to_none=True)

    # backpropagation
    loss.backward()
    optimizer.step()
print(loss.item())

4.400284290313721


In [33]:
# create initial tensor
tensor_empty_1x1 = torch.zeros((1, 1), dtype=torch.long)
print(f"Initial tensor: {tensor_empty_1x1}\n")

# generate tokens
generated_tensor = m.generate(idx = tensor_empty_1x1, max_new_tokens=500)
print(f"generated tensor shape: {generated_tensor.shape}")
print(f"generated tensor first 10 tokens: {generated_tensor[0][:10]}\n")

# extract first sequence from batch
first_sequence = generated_tensor[0]
print(f"first sequence shape: {first_sequence.shape}\n")

# convert to python list
token_list = first_sequence.tolist()
print(f"Token list lenght: {len(token_list)}")

# decode tokens to text
decoded_text = decode(token_list)

print(f"The final generated tokens are:")
print(decoded_text)

Initial tensor: tensor([[0]])

generated tensor shape: torch.Size([1, 501])
generated tensor first 10 tokens: tensor([ 0, 57, 22, 27, 24,  2, 40,  5, 18, 46])

first sequence shape: torch.Size([501])

Token list lenght: 501
The final generated tokens are:

sJOL!b'FhkgHj?iAlwNlMNlQKdt,oX!eyfsQQnquTRft-AqI, IGyx&ICyYea!cDvXWEKD!SD&-xx.yWglYuNU3EuK;whRz&fHVmHquZKic KDnmhwsEQff:wnOrWcC$FrW,Q'h3hhDW&lmj KZjnR:JLpm
g!-Zm'dZGbleBPzX-:$:LU:MKdWHWiJdgeBMGvrdcLFxIR:&oB :wolws?mGx'Gawaxh&-kn:'LBhRZwK'NeTuHqY'eIe!?ee.JqY-NQ'RNBxhvPoONDDYLUteM-x.;.MmEIMpgE,ihkKRScutNj:$:aqIq'cNs'ZhwtMxtK'C$Y$KSYuzlG?Iyft!$EKkyNOSIZ
b'& krV!aYVE!Stm.Mi
jfwgIEPuevsMQ MuVO McCCK,BhOgiIFOXnStOP?UnPerjHV'XgWJ
IR;qvEy.,N'cwj WP;P;:aRzPl
dDDDDQmMcysg!PvZ$BxmvSJyk:SHW Bg: DhDOIPT.OSt?


## The mathematical trick in self-attention

Toy example illustrating how matrx multiplication can be used for a weighted aggregation.

In [20]:
torch.manual_seed(42)

In [21]:
# create a lower triangular matrix
a = torch.tril(torch.ones(3, 3))
print(f"Initial lower triangular matrix:\n {a}")

Initial lower triangular matrix:
 tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


In [22]:
# Normalize rows to create weights
a = a / torch.sum(a, 1, keepdim=True)
print("Row-normalized weights:")
print(a)

Row-normalized weights:
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])


In [23]:
# create data matrix
b = torch.randint(0, 10, (3,2)).float()
print(f"Data matrix:\n {b}")

Data matrix:
 tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])


In [24]:
# perform weighted aggregation
c = a @ b # Matrix multiplication with @
print(f"Weighted aggregaion results:\n {c}")

Weighted aggregaion results:
 tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Matrix a acts as attention weights:
 - Row 1: [1.0, 0.0, 0.0] → "Only pay attention to position 1"
 - Row 2: [0.5, 0.5, 0.0] → "Average positions 1-2 equally"
 - Row 3: [0.33, 0.33, 0.33] → "Average all three positions equally"

Matrix b contains the data we want to aggregate

Matrix c is weighted combination

### Three diferent ways to implement computing a running average, building block attention mechanisms

In [25]:
# setup and goal
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C) # random input data
x.shape # 4 sequences, each with 8 timesteps, 2 features each

torch.Size([4, 8, 2])

In [26]:
# VERSION 1: explicit loops naive approach
xbow = torch.zeros((B,T,C))
for b in range(B): # for each batch
    for t in range(T): # for each time position
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)
        # result
        # mean(x[b,0])
        # mean(x[b,0], x[b,1])
        # mean(x[b,0], x[b,1], x[b,2])
        # ...

In [29]:
# VERSION 2: Matrix multiplication
wei = torch.tril(torch.ones(T,T)) # lower triangular matrix
# wei looks like:
# [[1., 0., 0., 0., 0., 0., 0., 0.],
#  [1., 1., 0., 0., 0., 0., 0., 0.],
#  [1., 1., 1., 0., 0., 0., 0., 0.],
#  [1., 1., 1., 1., 0., 0., 0., 0.],
#  [1., 1., 1., 1., 1., 0., 0., 0.],
#  [1., 1., 1., 1., 1., 1., 0., 0.],
#  [1., 1., 1., 1., 1., 1., 1., 0.],
#  [1., 1., 1., 1., 1., 1., 1., 1.]]

wei = wei / wei.sum(1, keepdim=True) # normalize each row
# After normalization:
# [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000], # 100% position 0
#  [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000], # 50% for pos1 and pos2
#  [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000], # 33% for pos1, pos2, pos3
#  [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000], # ...
#  ...
#  [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]]

xbow2 = wei @ x # (B,T,T) @ (B,T,C) -------> (B,T,C)  Matrix multiplication performs the weighted sum
torch.allclose(xbow, xbow2)

False

In [32]:
# VERSION 3: use softmax, most general solution
tril = torch.tril((torch.ones(T,T)))
# # lower triangular mask. Example for T=4:
# [[1, 0, 0, 0],
#  [1, 1, 0, 0],
#  [1, 1, 1, 0],
#  [1, 1, 1, 1]]

wei = torch.zeros((T,T)) # starts with zeros

wei = wei.masked_fill(tril == 0, float("-inf")) # set upper triangle to -oo
# wei now looks like:
# [[ 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
#  [ 0.,  0., -inf, -inf, -inf, -inf, -inf, -inf],
#  [ 0.,  0.,  0., -inf, -inf, -inf, -inf, -inf],
#  ...]

wei = F.softmax(wei, dim=-1)  # Softmax converts to probabilities
# After softmax:
# [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#  [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#  [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
#  ...]

In [33]:
xbow3 = wei @ x

# verification of the match with other implementation
torch.allclose(xbow, xbow3)

False

In [34]:
# VERSION 4: self attention
B,T,C = 4,8,32 # 4 sequences, 8 tokens each, 32-dimensional embeddings

# simulation of input data
x = torch.randn(B,T,C)

# attention head will produce 16-dimensional embeddings
head_size = 16

In [35]:
# three linear projections...
# What do I contain?
key = nn.Linear(C, head_size, bias=False)
# projects 32 ----> 16

# What am I interested in?
query = nn.Linear(C, head_size, bias=False)
# projects 32 ----> 16

# What information will I actually pass along?
value = nn.Linear(C, head_size, bias=False)
# projects 32 ----> 16

k = key(x)  # B, T, 16
q = query(x)  # B, T, 16
v = value(x)  # B, T, 16

In [36]:
# for each token, compute similarity with all other tokens
# High Score --> query matches that key well
# this is called DOT-PRODUCT ATTENTION
wei = q @ k.transpose(-2, -1) #B, T, T

In [37]:
# causal masking (decoder style)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))

In [38]:
wei = F.softmax(wei, dim=-1)
out = wei @ v

See the results

In [39]:
out.shape

torch.Size([4, 8, 16])

In [40]:
k.var()

tensor(0.3033, grad_fn=<VarBackward0>)

In [41]:
q.var()

tensor(0.3459, grad_fn=<VarBackward0>)

In [42]:
wei.var()

tensor(0.0493, grad_fn=<VarBackward0>)

In [44]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [45]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [46]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [47]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [48]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

The final train process